In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# !pip install torch-geometric-signed-directed

In [0]:
%pip install --upgrade networkx

In [0]:
dbutils.library.restartPython()

In [0]:
import logging
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

In [0]:
import sys

# Add sources directory to Python path
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
sys.path.insert(0, str(sources_path))

In [0]:
import logging

logger = logging.getLogger()  # root logger
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

In [0]:
"""
Retweet Prediction GNN Pipeline
================================
Each sample is a subgraph centered on a (user, tweet) pair.
Node features come from pretrained MagNet embeddings (.pt file).

Graph structure per sample:
  - Central user node
  - 2-hop neighbor user nodes
  - A binary node feature flag: did this neighbor retweet the tweet?

Target: did the central user retweet the tweet? (binary classification)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv

In [0]:
import torch
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {props.name}, free={free/1e9:.1f}GB / {total/1e9:.1f}GB")


In [0]:
import subprocess

def get_free_memory_per_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    return [int(x) for x in result.stdout.strip().split("\n")]

def get_best_gpu():
    free_mem = get_free_memory_per_gpu()
    best_gpu = max(range(len(free_mem)), key=lambda i: free_mem[i])
    print(f"Free memory per GPU (MiB): {free_mem}")
    print(f"Selected GPU {best_gpu} with {free_mem[best_gpu]} MiB free")
    return best_gpu

device = torch.device(f"cuda:{get_best_gpu()}")
# model = model.to(device)

In [0]:
# ---------------------------------------------------------------------------
# 1. Pretrained Embedding Lookup
# ---------------------------------------------------------------------------

class PretrainedEmbeddingLookup(nn.Module):
    """
    Maps global user IDs to their pretrained MagNet embeddings.
    Embeddings are frozen (not trained).
    """
    def __init__(self, embeddings_path: str, device: str):
        super().__init__()
        # Shape: [num_users, embedding_dim]
        pretrained = torch.load(embeddings_path, weights_only=True, map_location=device)
        # Register as a buffer so it moves with .to(device) but is not a parameter
        self.register_buffer("embeddings", pretrained)
        self.embedding_dim = pretrained.shape[1]

    def forward(self, user_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            user_ids: [N] global user IDs (indices into the embedding matrix)
        Returns:
            [N, embedding_dim] pretrained embeddings
        """
        return self.embeddings[user_ids]

In [0]:
# ---------------------------------------------------------------------------
# 2. Dataset
# ---------------------------------------------------------------------------
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class RetweetDataset(Dataset):
    """
    Each sample describes the local subgraph around a (central_user, tweet) pair.

    Expected input per sample (raw_samples list):
    {
        "central_user_id":  int,
        "neighbor_ids":     List[int],          # 2-hop neighbors (global user IDs)
        "retweeted_ids":    List[int],           # subset of neighbor_ids that retweeted
        "edge_index":       List[Tuple[int,int]],# edges as LOCAL node index pairs
        "label":            int,                 # 1 = central user retweeted, 0 = did not
    }

    Node ordering convention:
        index 0           → central user
        index 1..N-1      → neighbor users (in the order given by neighbor_ids)
    """

    def __init__(self, raw_samples: list):
        super().__init__()
        self.samples = raw_samples

    def len(self):
        return len(self.samples)

    def get(self, idx):
        s = self.samples[idx]

        # All node IDs in order: central first, then neighbors
        all_ids = [s["central_user_id"]] + list(s["neighbor_ids"])
        num_nodes = len(all_ids)

        user_ids = torch.tensor(all_ids, dtype=torch.long)

        # Binary retweet flag feature for each node
        retweeted_set = set(s["retweeted_ids"])
        retweet_flag = torch.tensor(
            [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
            dtype=torch.float
        ).unsqueeze(1)  # [N, 1]

        # Edge index (local indices)
        if len(s["edge_index"]) > 0:
            edge_index = torch.tensor(s["edge_index"], dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)

        label = torch.tensor(s["label"], dtype=torch.long)

        return Data(
            user_ids=user_ids,          # [N]   for embedding lookup
            retweet_flag=retweet_flag,  # [N,1] extra structural feature
            edge_index=edge_index,      # [2,E]
            y=label,                    # scalar
            num_nodes=num_nodes,
            central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(0, torch.tensor([0]), True)
        )


In [0]:
# ---------------------------------------------------------------------------
# 3. Model
# ---------------------------------------------------------------------------

class RetweetGNN(nn.Module):
    """
    Architecture:
        1. Pretrained embedding lookup  (frozen)
        2. Feed-forward projection      (trainable, adds retweet_flag)
        3. MagNetConv layers            (directed message passing via magnetic Laplacian)
        4. TransformerConv layer        (attention-based aggregation)
        5. Readout: central node repr   (not global pool — we care about node 0)
        6. MLP classifier + softmax
    """

    def __init__(
        self,
        embeddings_path: str,
        device: str,
        ff_hidden_dim: int = 256,
        gcn_hidden_dim: int = 128,
        transformer_dim: int = 128,
        transformer_heads: int = 4,
        num_classes: int = 2,
        dropout: float = 0.3,
        q: float = 0.25,    # MagNet phase parameter: controls directional sensitivity
        K: int = 1,         # Chebyshev order for MagNetConv
    ):
        super().__init__()

        # Multiply retweet flags by a scalar (learnable) so it's not dwarfed:
        self.flag_scale = nn.Parameter(torch.tensor(10.0))

        # 1. Frozen pretrained embeddings
        self.lookup = PretrainedEmbeddingLookup(embeddings_path, device)
        embed_dim = self.lookup.embedding_dim  # e.g. 128 from MagNet

        # Input to FF: embedding + retweet_flag (1 dim)
        ff_input_dim = embed_dim + 1

        # 2. Feed-forward projection (makes embeddings trainable/adaptable)
        self.ff = nn.Sequential(
            nn.Linear(ff_input_dim, ff_hidden_dim),
            nn.LayerNorm(ff_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden_dim, gcn_hidden_dim),
            nn.LayerNorm(gcn_hidden_dim),
            nn.GELU(),
        )

        # 3. MagNetConv layers
        # Each layer takes (x_real, x_imag) and returns (x_real, x_imag).
        # The imaginary stream carries directional phase information throughout.
        self.magnet1 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)
        # self.magnet2 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)

        # 4. Transformer layer
        self.transformer = TransformerConv(
            in_channels=gcn_hidden_dim * 2,  # real + imag concatenated
            out_channels=transformer_dim // transformer_heads,
            heads=transformer_heads,
            edge_dim=1,
            dropout=dropout,
            concat=True,   # output dim = transformer_dim
        )

        # LayerNorm on transformer output to prevent it from drowning shortcuts
        self.post_transformer_norm = nn.LayerNorm(transformer_dim)

        # 5. Residual gating: blend GNN features with shortcut features
        # Gate initialized to ~0 (sigmoid(-5) ≈ 0.007) so model starts from shortcuts
        self.gate_param = nn.Parameter(torch.tensor(-5.0))

        # Shortcut branch: small MLP on [rt_frac, neighbor_count]
        self.shortcut_head = nn.Sequential(
            nn.Linear(2, 16),
            nn.GELU(),
            nn.Linear(16, num_classes),
        )

        # GNN branch: classifier on normalized transformer output
        self.gnn_head = nn.Sequential(
            nn.Linear(transformer_dim, transformer_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(transformer_dim // 2, num_classes),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        user_ids     = data.user_ids       # [N_total]
        retweet_flag = data.retweet_flag   # [N_total, 1]
        edge_index   = data.edge_index     # [2, E_total]
        batch        = data.batch          # [N_total] — PyG batch vector
        central_mask = data.central_mask   # [N_total] bool, True for node 0 per graph

        # 1. Lookup pretrained embeddings (no grad)
        with torch.no_grad():
            pretrained = self.lookup(user_ids)  # [N, embed_dim]

        # 2. Concatenate retweet flag and project
        x = torch.cat([pretrained, self.flag_scale * retweet_flag], dim=-1)  # [N, embed_dim+1]
        x = self.ff(x)                                      # [N, gcn_hidden_dim]

        # 3. MagNetConv layers
        # MagNetConv signature: forward(x_real, x_imag, edge_index) -> (x_real, x_imag)
        # We start with x as the real part and zeros as the imaginary part.
        # The imaginary stream accumulates directional phase information across layers.
        x_real, x_imag = x, torch.zeros_like(x)

        x_real_res, x_imag_res = x_real, x_imag
        x_real, x_imag = self.magnet1(x_real, x_imag, edge_index)
        x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)
        x_real, x_imag = self.dropout(x_real), self.dropout(x_imag)

        # x_real, x_imag = self.magnet2(x_real, x_imag, edge_index)
        # x_real = x_real + x_real_res   # residual on real stream
        # x_imag = x_imag + x_imag_res   # residual on imaginary stream
        # x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)

        # Merge real and imaginary into a single representation before TransformerConv.
        # Concatenation preserves both undirected structure (real) and
        # directional phase information (imaginary) for the attention layer.
        x = torch.cat([x_real, x_imag], dim=-1)  # [N, 2*gcn_hidden_dim]

        # 4. Transformer layer
        # build edge_attr from the target node's flag:
        # for each edge (src→dst), attach the dst node's retweet flag
        edge_attr = retweet_flag[data.edge_index[1]]   # [E, 1]
        x = self.transformer(x, edge_index, edge_attr=edge_attr) # [N, transformer_dim]
        x = F.gelu(x)

        # 5. Extract central node representation per graph in the batch
        x = self.post_transformer_norm(x)                   # normalize before readout
        central_x = x[central_mask]                         # [batch_size, transformer_dim]
        assert central_x.shape[0] == data.batch.max().item() + 1, "central_mask broken!"

        # 5b. Shortcut features: retweet fraction & neighbor count per graph
        # These give the classifier direct access to the strongest signal
        num_graphs = data.batch.max().item() + 1
        non_central = ~central_mask
        # retweet fraction per graph (mean of retweet_flag excluding central node)
        nc_flags = retweet_flag[non_central].squeeze()  # [num_non_central]
        nc_batch = batch[non_central]                    # [num_non_central]
        rt_sum = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, nc_flags)
        node_counts = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, torch.ones_like(nc_flags))
        rt_frac = (rt_sum / node_counts.clamp(min=1)).unsqueeze(-1)
        node_counts = (node_counts / 50.0).unsqueeze(-1)  # normalize

        shortcuts = torch.cat([rt_frac, node_counts], dim=-1)  # [batch_size, 2]

        # 6. Gated blend: shortcut logits dominate early, GNN logits blend in as gate opens
        gate = torch.sigmoid(self.gate_param)  # scalar in [0, 1]
        shortcut_logits = self.shortcut_head(shortcuts)
        gnn_logits = self.gnn_head(central_x)
        logits = (1 - gate) * shortcut_logits + gate * gnn_logits

        return logits  # raw logits; use cross_entropy in training


In [0]:
# ---------------------------------------------------------------------------
# 4. Training & Evaluation
# ---------------------------------------------------------------------------
from sklearn.metrics import f1_score

def soft_f1_loss(logits, labels, eps=1e-8):
    probs = F.softmax(logits, dim=-1)[:, 1]   # P(positive)
    tp = (probs * labels).sum()
    fp = (probs * (1 - labels)).sum()
    fn = ((1 - probs) * labels).sum()
    f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    return 1 - f1


def focal_loss(logits, labels, alpha=0.75, gamma=2.0):
    """Focal loss — down-weights easy negatives, focuses on hard positives."""
    ce = F.cross_entropy(logits, labels.long(), reduction='none')
    pt = torch.exp(-ce)
    # alpha weighting: higher weight for positive class
    alpha_t = alpha * labels + (1 - alpha) * (1 - labels)
    loss = alpha_t * (1 - pt) ** gamma * ce
    return loss.mean()


def combined_loss(logits, labels, class_weights, epoch, warmup_epochs=10):
    """Phase 1: weighted CE to break symmetry. Phase 2: blend in soft F1."""
    # Weighted cross-entropy (strong per-sample gradients)
    ce = F.cross_entropy(logits, labels.long(), weight=class_weights.to(logits.device))
    if epoch <= warmup_epochs:
        return ce
    # After warmup, blend soft F1 to directly optimize the metric
    sf1 = soft_f1_loss(logits, labels)
    return 0.5 * ce + 0.5 * sf1

def train_epoch(model, loader, optimizer, device, class_weights, epoch=1):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = combined_loss(logits, batch.y.float(), class_weights, epoch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    all_probs, all_labels, all_preds = [], [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = F.softmax(logits, dim=-1)[:, 1]
        preds = logits.argmax(dim=-1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(batch.y.cpu())
        all_preds.append(preds.cpu())
    acc = correct / total
    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    all_preds  = torch.cat(all_preds)

    f1 = f1_score(all_labels, all_preds)

    return acc, f1, all_probs, all_labels


def train_model(
    model: RetweetGNN,
    raw_train_samples: list,
    raw_val_samples: list,
    epochs: int = 50,
    batch_size: int = 32,
    lr: float = 1e-3,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    y = np.array([s["label"] for s in raw_train_samples])
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    train_ds = RetweetDataset(raw_train_samples)
    val_ds   = RetweetDataset(raw_val_samples)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # Only optimize non-frozen parameters
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"Training on {device} | {len(train_ds)} train / {len(val_ds)} val samples")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    best_val_f1 = 0
    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device, class_weights, epoch=epoch)
        val_acc, val_f1, _, _ = evaluate(model, val_loader, device)
        scheduler.step()

        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), "best_retweet_gnn.pt")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), "best_retweet_gnn.pt")


        if epoch % 5 == 0 or True:
            train_acc, train_f1, _, _ = evaluate(model, train_loader, device)

            gate_val = torch.sigmoid(model.gate_param).item()
            print(f"Epoch {epoch:>3} | Loss: {loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Best: {best_val_f1:.4f} | Gate: {gate_val:.4f}")

    print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")
    return model

In [0]:

# ---------------------------------------------------------------------------
# 5. Usage example
# ---------------------------------------------------------------------------

# if __name__ == "__main__":
# Example: construct a single sample manually
sample = {
    "central_user_id": 42,
    "neighbor_ids": [7, 15, 99, 204, 301],
    "retweeted_ids": [15, 99],          # these neighbors retweeted
    "edge_index": [                     # LOCAL indices (0=central, 1..5=neighbors)
        (0, 1), (1, 0),
        (0, 2), (2, 0),
        (1, 3), (3, 4),
        (0, 5), (5, 0),
    ],
    "label": 1,                         # central user DID retweet
}

train_samples = [sample] * 1000

val_samples = train_samples[:100]

In [0]:
# # Build dataset from your list of samples and train:
# model = build_and_train(
#     raw_train_samples=train_samples,
#     raw_val_samples=val_samples,
#     embeddings_path="node_embeddings.pt",
#     epochs=10,
#     device=device
# )

- Load one user and transform to data to the input format of the GNN model

In [0]:

# import json
# from random import sample
#
# with open("../../data/datasets/user_splits.json") as f:
#     user_splits = json.load(f)
# user_ids = sample(user_splits["u_train"], 10)
# uid = user_ids[0]
# uid

In [0]:
uid = 283762641
# F1 SVC
# "283762641": 0.63888888888888884

In [0]:
from utils import load_dataframe_raw
data_uid = load_dataframe_raw(uid, sparse=True)
X_tr, X_te, y_tr, y_te = data_uid

In [0]:
X_tr.shape, X_te.shape, y_tr.shape, y_te.shape

In [0]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import numpy as np

# Compute class imbalance ratio
neg = (y_tr == 0).sum()
pos = (y_tr == 1).sum()
ratio = neg / pos
print(f"Class distribution — Neg: {neg}, Pos: {pos}, scale_pos_weight: {ratio:.2f}\n")

# Split training set to create a validation set for early stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr
)

model = XGBClassifier(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.05,
    scale_pos_weight=ratio,
    reg_alpha=1.0,
    reg_lambda=5.0,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=50,
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(f"Best iteration: {model.best_iteration}")

# Retrain on full training set with optimal n_estimators
best_n = model.best_iteration
final_model = XGBClassifier(
    n_estimators=best_n,
    max_depth=3,
    learning_rate=0.05,
    scale_pos_weight=ratio,
    reg_alpha=1.0,
    reg_lambda=5.0,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
final_model.fit(X_tr, y_tr)

print(f"\nFinal model ({best_n} trees, full training set):")
print(f"Train F1: {f1_score(y_tr, final_model.predict(X_tr)):.4f}")
print(f"Test  F1: {f1_score(y_te, final_model.predict(X_te)):.4f}")

In [0]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# Sweep over C with balanced class weights
results = []
for C in [0.01, 0.02, 0.03, 0.05, 0.07, 0.1]:
    svc = LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=42)
    svc.fit(X_tr, y_tr)
    train_f1 = f1_score(y_tr, svc.predict(X_tr))
    test_f1 = f1_score(y_te, svc.predict(X_te))
    results.append((C, train_f1, test_f1))
    print(f"C={C:<6}  Train F1: {train_f1:.4f}  Test F1: {test_f1:.4f}  gap: {train_f1-test_f1:.4f}")

best = max(results, key=lambda x: x[2])
print(f"\nBest test F1: {best[2]:.4f} at C={best[0]}")

In [0]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score

from sklearn.metrics.pairwise import linear_kernel

# Convert sparse DataFrame to scipy CSR for efficient kernel computation
X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

# Precompute linear kernel matrices
K_train = linear_kernel(X_tr_sp)
K_test = linear_kernel(X_te_sp, X_tr_sp)

# Sweep over C with balanced class weights
results = []
for C in [0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.5]:
    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
    svc.fit(K_train, y_tr)
    train_f1 = f1_score(y_tr, svc.predict(K_train))
    test_f1 = f1_score(y_te, svc.predict(K_test))
    results.append((C, train_f1, test_f1))
    print(f"C={C:<5}  Train F1: {train_f1:.4f}  Test F1: {test_f1:.4f}  gap: {train_f1-test_f1:.4f}")

best = max(results, key=lambda x: x[2])
print(f"\nBest test F1: {best[2]:.4f} at C={best[0]}")

In [0]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel

# Convert sparse DataFrame to scipy CSR
X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

# Sweep over gamma and C with precomputed RBF kernel
results = []
for gamma in [0.05, 0.08, 0.1, 0.15, 0.2]:
    K_train = rbf_kernel(X_tr_sp, gamma=gamma)
    K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
    for C in [0.01, 0.05, 0.1, 0.2]:
        svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
        svc.fit(K_train, y_tr)
        train_f1 = f1_score(y_tr, svc.predict(K_train))
        test_f1 = f1_score(y_te, svc.predict(K_test))
        results.append((gamma, C, train_f1, test_f1))
        print(f"gamma={gamma:<6} C={C:<5}  Train F1: {train_f1:.4f}  Test F1: {test_f1:.4f}  gap: {train_f1-test_f1:.4f}")

best = max(results, key=lambda x: x[3])
print(f"\nBest test F1: {best[3]:.4f} at gamma={best[0]}, C={best[1]}")

## Baseline Comparison (user 283762641)

| Model | Best Params | Train F1 | Test F1 | Gap |
| --- | --- | --- | --- | --- |
| XGBoost (early stopping) | depth=3, lr=0.05, 330 trees, scale_pos_weight=4.72 | 0.4015 | 0.3879 | 0.014 |
| LinearSVC | C=0.05, balanced | 0.7028 | 0.4701 | 0.233 |
| Kernel SVC (linear, precomputed) | C=0.07, balanced | 0.4051 | 0.4022 | 0.003 |
| **Kernel SVC (RBF, precomputed)** | **gamma=0.1, C=0.2, balanced** | **0.7443** | **0.6370** | **0.107** |
| Reference: original SVC | — | — | 0.6389 | — |

In [0]:
from utils import create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH
import networkx as nx

In [0]:
nx.__version__

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
central_user_id = uid
train_gnn_samples, val_gnn_samples = create_gnn_train_val_samples(central_user_id, graph, X_tr, y_tr, X_te, y_te)

In [0]:
len(train_gnn_samples)

In [0]:
len(val_gnn_samples)

In [0]:
DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

In [0]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Reinitialize model for clean run
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device
).to(device)

# Train GNN with improved loss (weighted CE warmup -> soft F1 blend)
# Larger batch so soft F1 sees enough positives per batch
train_model(
    model=model,
    raw_train_samples=train_gnn_samples,
    raw_val_samples=val_gnn_samples,
    epochs=50,
    batch_size=128,
    device=device,
    lr=1e-2,
)